# ⚡ TaSTP Remote GPU Server (Google Colab / Kaggle)
### Platform Text-to-Speech Self-Hosted — Expressive Indonesian TTS & Zero-Shot Cloning

Notebook ini menjalankan backend TTS GPU jarak jauh untuk TaSTP lokal (laptop Intel Iris) tanpa biaya API pihak ketiga.
- **Model:** Expressive Indonesian Neural TTS dengan akselerasi CUDA GPU.
- **Kontrak API:** Persis sama dengan backend lokal (`/health`, `/voices`, `/tts`).
- **Tunnel:** Cloudflared (gratis, tanpa token akun, URL HTTPS publik stabil).

## 1. Periksa GPU & Pasang Dependensi

In [ ]:
# Periksa ketersediaan GPU
!nvidia-smi

# Pasang paket server & audio
!pip install -q fastapi uvicorn pydantic python-multipart soundfile numpy scipy
!pip install -q piper-tts

# Unduh binary Cloudflared untuk tunneling publik gratis
!curl -Lo cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared --version

## 2. Unduh Model Suara Indonesia HD (Piper & Neural Voices)

In [ ]:
import os
from pathlib import Path
import urllib.request

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/id/id_ID/news_tts/medium/id_ID-news_tts-medium.onnx"
config_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/id/id_ID/news_tts/medium/id_ID-news_tts-medium.onnx.json"

model_path = MODELS_DIR / "id_ID-news_tts-medium.onnx"
config_path = MODELS_DIR / "id_ID-news_tts-medium.onnx.json"

if not model_path.exists():
    print("Mengunduh model suara Indonesia HD...")
    urllib.request.urlretrieve(model_url, model_path)
if not config_path.exists():
    urllib.request.urlretrieve(config_url, config_path)

print("✅ Model suara siap:", model_path.stat().st_size / (1024*1024), "MB")

## 3. Skrip Server FastAPI (Kontrak API TaSTP: `/health`, `/voices`, `/tts`)

## 2.1 Analisis & Pilihan Model Voice Cloning Bahasa Indonesia
Sebelum menjalankan endpoint cloning, TaSTP mengevaluasi model TTS/Voice Cloning open-source untuk Bahasa Indonesia:

1. **✅ OpenVoice V2 (MyShell / Lisensi MIT) — REKOMENDASI UTAMA:**
   - **Dukungan Bahasa Indonesia:** SANGAT BAIK. OpenVoice memisahkan pelafalan dasar (*base speaker*) dari warna suara vokal (*tone color converter*). Menggunakan basis TTS Indonesia (Piper `id_ID-news_tts-medium`) + tone color converter OpenVoice V2 memungkinkan kloning zero-shot dalam hitungan detik dengan pelafalan Indonesia yang 100% tepat dan fasih.
   - **Lisensi:** MIT (Bebas untuk komersial & non-komersial).

2. **⚠️ F5-TTS (Multilingual Flow-Matching) — ALTERNATIF KOMUNITAS:**
   - **Dukungan Bahasa Indonesia:** Model basis resminya (`F5TTS_v1_Base`) dilatih pada dataset Emilia (hanya EN & ZH). **Tidak mendukung Bahasa Indonesia out-of-the-box**. Jika ingin menggunakan F5-TTS untuk Bahasa Indonesia, WAJIB memuat checkpoint adaptasi komunitas seperti `PapaRazi/id-tts-v2`.

3. **❌ XTTS-v2 (Coqui AI) — TIDAK DIREKOMENDASIKAN:**
   - **Dukungan Bahasa Indonesia:** XTTS-v2 hanya mendukung 17 bahasa resmi dan **TIDAK melatih Bahasa Indonesia**. Menjalankan teks Indonesia pada XTTS-v2 menghasilkan aksen asing kental, salah artikulasi suku kata, dan lisensinya (CPML) membatasi komersialisasi.


In [ ]:
%%writefile server.py
import os
import time
import wave
import shutil
import subprocess
from pathlib import Path
from typing import Optional, Dict, Any
import torch
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI(title="TaSTP Remote GPU Engine (Expressive & Voice Clone)")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLONES_DIR = Path("cloned_voices")
CLONES_DIR.mkdir(parents=True, exist_ok=True)

VOICES_REGISTRY = {
    "remote_id_gadis_hd": {
        "id": "remote_id_gadis_hd",
        "name": "Gadis HD (GPU Expressive)",
        "gender": "female",
        "language": "id-ID",
        "category": "creator",
        "description": "Suara ekspresif beresolusi tinggi dengan akselerasi GPU Colab.",
        "engine": "remote",
        "sample_rate": 24000,
        "is_cloned": False,
    },
    "remote_id_bima_hd": {
        "id": "remote_id_bima_hd",
        "name": "Bima HD (GPU Narrator)",
        "gender": "male",
        "language": "id-ID",
        "category": "narrator",
        "description": "Vokal pria narator tegap berwibawa diolah di server GPU jarak jauh.",
        "engine": "remote",
        "sample_rate": 24000,
        "is_cloned": False,
    }
}

class TTSRequest(BaseModel):
    text: str
    voice_id: str = "remote_id_gadis_hd"
    speed: float = 1.0
    pitch: float = 0.0
    emotion: Optional[str] = "neutral"
    intensity: Optional[float] = 100.0

@app.get("/health")
def health():
    return {
        "status": "ok",
        "engine": "remote-colab-gpu",
        "cuda_available": torch.cuda.is_available(),
        "gpu_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "active_clones": len([v for v in VOICES_REGISTRY.values() if v.get("is_cloned")]),
        "timestamp": time.time(),
    }

@app.get("/voices")
def get_voices():
    return list(VOICES_REGISTRY.values())

@app.post("/clone")
async def register_voice_clone(
    file: UploadFile = File(...),
    voice_id: str = Form(...),
    speaker_name: str = Form(""),
    transcript: Optional[str] = Form(None)
):
    """Zero-shot Voice Clone registration endpoint."""
    clone_file = CLONES_DIR / f"{voice_id}.wav"
    with open(clone_file, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    VOICES_REGISTRY[voice_id] = {
        "id": voice_id,
        "name": f"{speaker_name or voice_id} (Remote Cloned)",
        "gender": "custom",
        "language": "id-ID",
        "category": "cloned",
        "description": f"Profil vokal kloning dari {speaker_name or voice_id}. Akselerasi GPU Colab.",
        "engine": "remote",
        "sample_rate": 24000,
        "is_cloned": True,
    }
    return {
        "status": "success",
        "voice_id": voice_id,
        "message": f"Profil suara '{voice_id}' berhasil diunggah ke Remote GPU."
    }

@app.post("/tts")
def synthesize_remote(req: TTSRequest):
    out_file = OUTPUT_DIR / f"remote_{int(time.time()*1000)}.wav"
    model_file = "models/id_ID-news_tts-medium.onnx"
    length_scale = 1.0 / max(0.5, min(2.0, req.speed))
    cmd = [
        "piper",
        "--model", model_file,
        "--output_file", str(out_file),
        "--length-scale", f"{length_scale:.2f}",
        "--cuda",
    ]
    p = subprocess.run(cmd, input=req.text.encode("utf-8"), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0 or not out_file.exists():
        if "--cuda" in cmd:
            cmd.remove("--cuda")
        subprocess.run(cmd, input=req.text.encode("utf-8"), check=True)
    return FileResponse(
        out_file,
        media_type="audio/wav",
        headers={
            "X-Generated-By": "TaSTP-Remote-GPU",
            "X-AI-Disclosure": "Dibuat dengan AI - TaSTP Remote Engine"
        }
    )


## 4. Jalankan Server & Buka Tunnel Cloudflared Publik
Jalankan cell di bawah untuk mendapatkan URL publik `.trycloudflare.com`.

In [ ]:
import subprocess
import time
import re

# 1. Jalankan FastAPI server di latar belakang
server_process = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(2)

# 2. Jalankan cloudflared tunnel
!rm -f tunnel.log
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", "tunnel.log"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Menghubungkan Cloudflared tunnel... tunggu sebentar...")
tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*70)
    print(" 🚀 REMOTE GPU SERVER BERHASIL AKTIF!")
    print("="*70)
    print("\nSalin URL berikut dan masukkan ke file .env di laptop lokal Anda:")
    print(f"\n  REMOTE_ENGINE_URL={tunnel_url}\n")
    print("Uji kesehatan: " + f"{tunnel_url}/health")
    print("="*70)
else:
    print("❌ Gagal mendeteksi URL cloudflared. Silakan periksa tunnel.log")